In [13]:
import pandas as pd

folder_2020 = r'C:\Users\sahil\projects\boston-mbta-ridership\data\raw\Arrival_Departure\2020' + '\\'
processed   = r'C:\Users\sahil\projects\boston-mbta-ridership\data\processed'

# Columns we need
keep = ['service_date', 'route_id', 'direction', 'half_trip_id',
        'time_point_id', 'point_type', 'scheduled', 'actual']

q1 = pd.read_csv(folder_2020 + 'Bus Arrival Departure Times Jan-Mar 2020.csv',  usecols=keep, dtype=str)
q2 = pd.read_csv(folder_2020 + 'Bus Arrival Departure Times Apr-June 2020.csv', usecols=keep, dtype=str)
q3 = pd.read_csv(folder_2020 + 'Bus Arrival Departure Times Jul-Sep 2020.csv',  usecols=keep, dtype=str)
print(q1.shape, q2.shape, q3.shape)

(7215761, 8) (5901432, 8) (6080638, 8)


In [14]:
# Oct-Dec uses different column names, so rename them to match the other quarters
q4_names = {
    'ServiceDate': 'service_date',
    'Route':       'route_id',
    'Direction':   'direction',
    'HalfTripId':  'half_trip_id',
    'Timepoint':   'time_point_id',
    'PointType':   'point_type',
    'Scheduled':   'scheduled',
    'Actual':      'actual',
}

q4 = pd.read_csv(folder_2020 + 'Bus Arrival Departure Times Oct-Dec 2020.csv',
                 usecols=list(q4_names.keys()), dtype=str)
q4 = q4.rename(columns=q4_names)

print(q4.shape)
print(q4['scheduled'].head(3).tolist()) 

(5782659, 8)
['1900-01-01 05:05:00.000', '1900-01-01 05:07:00.000', '1900-01-01 05:10:00.000']


In [15]:
ad20 = pd.concat([q1, q2, q3, q4], ignore_index=True)
print(ad20.shape)

(24980490, 8)


In [16]:
ad20['route_id'] = ad20['route_id'].str.strip().str.lstrip('0')     # '01' -> '1', to match ridership

ad20['service_date'] = pd.to_datetime(ad20['service_date'], format='mixed', errors='coerce')
ad20['scheduled']    = pd.to_datetime(ad20['scheduled'],    format='mixed', errors='coerce')
ad20['actual']       = pd.to_datetime(ad20['actual'],       format='mixed', errors='coerce')

ad20 = ad20.dropna(subset=['service_date', 'scheduled', 'actual'])   # drop rows with missing times
print(ad20.shape)

(23948975, 8)


In [17]:
# Lateness in minutes (negative = early)
ad20['lateness_min'] = (ad20['actual'] - ad20['scheduled']).dt.total_seconds() / 60

# Day type from the service date (Monday=0 ... Sunday=6)
ad20['day_type'] = 'weekday'
ad20.loc[ad20['service_date'].dt.dayofweek == 5, 'day_type'] = 'saturday'
ad20.loc[ad20['service_date'].dt.dayofweek == 6, 'day_type'] = 'sunday'

# Hour of the day (times use a fake 1900-01-01 date; past midnight goes above 24)
hour = (ad20['scheduled'] - pd.Timestamp('1900-01-01')).dt.total_seconds() / 3600

# Peak = weekdays, 7-9 AM and 4-7 PM
peak_hour = ((hour >= 7) & (hour < 9)) | ((hour >= 16) & (hour < 19))
ad20['period'] = 'off_peak'
ad20.loc[peak_hour & (ad20['day_type'] == 'weekday'), 'period'] = 'peak'

In [18]:
# Sort buses by scheduled time at each stop, then measure the gap to the bus before
ad20 = ad20.sort_values(['service_date', 'route_id', 'direction', 'time_point_id', 'scheduled'])
groups = ad20.groupby(['service_date', 'route_id', 'direction', 'time_point_id'])

ad20['sched_gap']  = groups['scheduled'].diff().dt.total_seconds() / 60
ad20['actual_gap'] = groups['actual'].diff().dt.total_seconds() / 60

# Bunched = actual gap less than half the scheduled gap (1 = yes, 0 = no)
ad20['bunched'] = (ad20['actual_gap'] < 0.5 * ad20['sched_gap']).astype(float)
ad20.loc[ad20['sched_gap'].isna(), 'bunched'] = float('nan')   # first bus of the day has no gap

In [19]:
# Trip time = actual time at the end point minus actual time at the start point
starts = ad20[ad20['point_type'] == 'Startpoint']
ends   = ad20[ad20['point_type'] == 'Endpoint']

trip_keys = ['service_date', 'route_id', 'direction', 'half_trip_id']

trips = starts[trip_keys + ['actual', 'day_type', 'period']].merge(
            ends[trip_keys + ['actual']], on=trip_keys, suffixes=('_start', '_end'))

trips['trip_time_min'] = (trips['actual_end'] - trips['actual_start']).dt.total_seconds() / 60
trips = trips[(trips['trip_time_min'] > 0) & (trips['trip_time_min'] < 300)]   # drop impossible values
print(trips.shape)

(2946979, 9)


In [ ]:
ad20['month']  = ad20['service_date'].dt.month
trips['month'] = trips['service_date'].dt.month

keys = ['route_id', 'month', 'day_type', 'period']

# Lateness and bunching per route, month, day type and period
timing = ad20.groupby(keys, as_index=False).agg(
    avg_lateness_min=('lateness_min', 'mean'),
    pct_bunched=('bunched', 'mean'),
    n_observations=('lateness_min', 'size'))

# Trip times per route, month, day type and period
trip_times = trips.groupby(keys, as_index=False).agg(
    median_trip_time_min=('trip_time_min', 'median'),
    n_trips=('trip_time_min', 'size'))

service20 = timing.merge(trip_times, on=keys, how='left')
service20['year'] = 2020
service20['fare_free'] = service20['route_id'].isin(['23', '28', '29'])



In [25]:
ad_folder = r'C:\Users\sahil\projects\boston-mbta-ridership\data\raw\Arrival_Departure'
processed = r'C:\Users\sahil\projects\boston-mbta-ridership\data\processed'

# Columns we need (2021+ files all use these names)
keep = ['service_date', 'route_id', 'direction_id', 'half_trip_id',
        'time_point_id', 'point_type', 'scheduled', 'actual']

In [26]:
def clean_month(year, month):
    """Clean one monthly file and return a summary per route, day type and period.
    Same steps as the 2020 cells, plus a time-zone fix."""

    # ---- Load ----
    file_name = f'MBTA-Bus-Arrival-Departure-Times_{year}-{month:02d}.csv'   # e.g. ..._2023-04.csv
    df = pd.read_csv(ad_folder + '\\' + str(year) + '\\' + file_name, usecols=keep, dtype=str)

    # ---- Fix data types ----
    df['route_id'] = df['route_id'].str.strip().str.lstrip('0')            # '01' -> '1'
    df['service_date'] = pd.to_datetime(df['service_date'], errors='coerce')
    df['scheduled'] = pd.to_datetime(df['scheduled'], utc=True, errors='coerce').dt.tz_localize(None)
    df['actual']    = pd.to_datetime(df['actual'],    utc=True, errors='coerce').dt.tz_localize(None)
    df = df.dropna(subset=['service_date', 'scheduled', 'actual'])

    # ---- Lateness and day type ----
    df['lateness_min'] = (df['actual'] - df['scheduled']).dt.total_seconds() / 60

    dow = df['service_date'].dt.dayofweek
    df['day_type'] = 'weekday'
    df.loc[dow == 5, 'day_type'] = 'saturday'
    df.loc[dow == 6, 'day_type'] = 'sunday'

    # ---- Peak period ----
    # These files store times in UTC. Boston is 4 hours behind in summer, 5 in winter.
    utc_offset = 4 if 4 <= month <= 10 else 5
    hour = (df['scheduled'] - pd.Timestamp('1900-01-01')).dt.total_seconds() / 3600 - utc_offset

    peak_hour = ((hour >= 7) & (hour < 9)) | ((hour >= 16) & (hour < 19))
    df['period'] = 'off_peak'
    df.loc[peak_hour & (df['day_type'] == 'weekday'), 'period'] = 'peak'

    # ---- Bunching ----
    df = df.sort_values(['service_date', 'route_id', 'direction_id', 'time_point_id', 'scheduled'])
    groups = df.groupby(['service_date', 'route_id', 'direction_id', 'time_point_id'])

    df['sched_gap']  = groups['scheduled'].diff().dt.total_seconds() / 60
    df['actual_gap'] = groups['actual'].diff().dt.total_seconds() / 60
    df['bunched'] = (df['actual_gap'] < 0.5 * df['sched_gap']).astype(float)
    df.loc[df['sched_gap'].isna(), 'bunched'] = float('nan')

    # ---- Trip time ----
    trip_keys = ['service_date', 'route_id', 'direction_id', 'half_trip_id']
    has_id = df.dropna(subset=['half_trip_id'])          # blank IDs would pair up wrongly
    starts = has_id[has_id['point_type'] == 'Startpoint']
    ends   = has_id[has_id['point_type'] == 'Endpoint']

    trips = starts[trip_keys + ['actual', 'day_type', 'period']].merge(
                ends[trip_keys + ['actual']], on=trip_keys, suffixes=('_start', '_end'))
    trips['trip_time_min'] = (trips['actual_end'] - trips['actual_start']).dt.total_seconds() / 60
    trips = trips[(trips['trip_time_min'] > 0) & (trips['trip_time_min'] < 300)]

    # ---- Summarize ----
    keys = ['route_id', 'day_type', 'period']

    timing = df.groupby(keys, as_index=False).agg(
        avg_lateness_min=('lateness_min', 'mean'),
        pct_bunched=('bunched', 'mean'),
        n_observations=('lateness_min', 'size'))

    trip_times = trips.groupby(keys, as_index=False).agg(
        median_trip_time_min=('trip_time_min', 'median'),
        n_trips=('trip_time_min', 'size'))

    summary = timing.merge(trip_times, on=keys, how='left')
    summary['year'] = year
    summary['month'] = month
    summary['fare_free'] = summary['route_id'].isin(['23', '28', '29'])
    return summary

print('Function loaded')

Function loaded


In [27]:
results = []

for year in [2021, 2022, 2023, 2024, 2025]:
    for month in range(1, 13):
        results.append(clean_month(year, month))
    print(year, 'done')

2021 done
2022 done
2023 done
2024 done
2025 done


In [32]:
service_2021_2025 = pd.concat(results, ignore_index=True)
service = pd.concat([service20, service_2021_2025], ignore_index=True)

service = service.round(2)

service = service.sort_values(['route_id', 'year', 'month', 'day_type', 'period']).reset_index(drop=True)

service.to_csv(processed + r'\service_route_month.csv', index=False)